# Chapter 3 — Claude Agent SDK: Building Agentic Systems · Practice

**Goal:** write the agentic loop by hand once, then let the Claude Agent SDK own it, and build a coordinator with subagents, custom tools, and hooks that enforce policy in code.

- Theory: [README.md](README.md) (read the matching section before each part)
- Estimated time: 3–4 hours
- Estimated API cost: **$0** for offline cells; about **$0.50–$1.50** if you run every live cell with the default model. Every Agent SDK run in this notebook is hard-capped with `max_budget_usd`.
- Prerequisites: [Chapter 1](../01-claude-api-fundamentals/README.md) and [Chapter 2](../02-tool-use/README.md), the repo venv, and a `.env` with `ANTHROPIC_API_KEY` (see [00-prerequisites](../../00-prerequisites/README.md))

**How this notebook works**

- Cells marked **(offline)** run without a key or network. They build configs, call tool handlers directly, and replay scripted responses, so you can learn the control flow for free.
- Cells marked **(live)** call the API. Each one calls `require_live()` first, so the Agent SDK never starts without `ANTHROPIC_API_KEY`. (The SDK bundles Claude Code, which could otherwise fall back to a logged-in Claude account.)
- Jupyter already runs an event loop, so use top-level `await` (as these cells do), never `asyncio.run(...)`.

In [ ]:
# Setup (offline)
import asyncio
import copy
import dataclasses
import json
import os
import random
import re
import tempfile
from datetime import datetime, timezone
from itertools import count
from types import SimpleNamespace
from typing import Any

import anthropic
from anthropic.types import Message, Usage
from anthropic.types import TextBlock as ApiTextBlock, ToolUseBlock as ApiToolUseBlock
from claude_agent_sdk import (
    AgentDefinition, AssistantMessage, ClaudeAgentOptions, ClaudeSDKClient, HookMatcher,
    PermissionResultAllow, PermissionResultDeny, ResultMessage, SystemMessage, TextBlock, ToolAnnotations,
    ToolPermissionContext, ToolResultBlock, ToolUseBlock, UserMessage, create_sdk_mcp_server, list_sessions,
    query, rename_session, tool,
)
from dotenv import find_dotenv, load_dotenv

load_dotenv(find_dotenv())
MODEL = os.getenv("CLAUDE_MODEL", "claude-sonnet-5-5")
SUBAGENT_MODEL = "haiku"          # alias accepted by AgentDefinition.model; cheap model for narrow roles
client = anthropic.Anthropic()    # the key is only needed when a live cell actually calls it
SANDBOX_DIR = tempfile.mkdtemp(prefix="ch3-agent-")   # live agents run here, never in your repo


def require_live() -> None:
    """Guard for live cells: refuse to start the Agent SDK without an explicit API key."""
    if not os.getenv("ANTHROPIC_API_KEY"):
        raise RuntimeError("Live cell skipped: set ANTHROPIC_API_KEY in .env (see 00-prerequisites).")


print("model:", MODEL, "| subagent model:", SUBAGENT_MODEL, "| agent cwd:", SANDBOX_DIR)

In [ ]:
# Key check (skip this cell to stay offline)
assert os.getenv("ANTHROPIC_API_KEY"), "Set ANTHROPIC_API_KEY in .env (see 00-prerequisites)"
print("API key found: live cells are enabled.")

## 3.1 The agentic loop

An agent is a model running in a loop: call Claude, run the tools it asks for, append the results, and call it again until `stop_reason == "end_turn"`. With the Client SDK **you** own that loop; with the Agent SDK, `query()` runs it and streams messages ending in a `ResultMessage` whose `subtype` tells you how it ended. Iteration caps and text matching are guardrails or bugs, never the completion signal. [Theory](README.md#31-the-agentic-loop)

### Mock tools for the manual loop

Two read-only tools over in-memory data. Chapter 2 covered the schema format; here we only care about the loop around them.

In [ ]:
# Mock data + Client SDK tool schemas (offline)
CUSTOMERS = {"ana@example.com": {"customer_id": "C-7", "name": "Ana"}}
ORDERS = {
    "A-1001": {"customer_id": "C-7", "status": "shipped", "total_usd": 129.0},
    "A-1002": {"customer_id": "C-7", "status": "delivered", "total_usd": 940.0},
}


def get_customer(email: str) -> dict:
    return CUSTOMERS.get(email.lower(), {"result": "no_match", "email": email})


def lookup_order(order_id: str) -> dict:
    return ORDERS.get(order_id, {"result": "no_match", "order_id": order_id})


IMPLS = {"get_customer": get_customer, "lookup_order": lookup_order}
TOOLS = [
    {
        "name": "get_customer",
        "description": "Find a customer by email. Call this first to verify identity.",
        "input_schema": {"type": "object", "properties": {"email": {"type": "string"}}, "required": ["email"]},
    },
    {
        "name": "lookup_order",
        "description": "Get status and total (USD) for one order ID such as A-1001.",
        "input_schema": {"type": "object", "properties": {"order_id": {"type": "string"}}, "required": ["order_id"]},
    },
]
print([t["name"] for t in TOOLS])

In [ ]:
# The manual loop (offline definition; works with a real or a fake client)
def text_of(resp) -> str:
    return "".join(b.text for b in resp.content if b.type == "text")


def run_agent(llm, task: str, tools: list[dict], impls: dict, *, safety_cap: int = 10, log: bool = True):
    """Client SDK agentic loop: stop on end_turn, run tools on tool_use, fail loudly on anything else."""
    messages: list[dict] = [{"role": "user", "content": task}]
    for turn in range(1, safety_cap + 1):
        resp = llm.messages.create(model=MODEL, max_tokens=512, tools=tools, messages=messages)
        if log:
            calls = [f"{b.name}({json.dumps(b.input)})" for b in resp.content if b.type == "tool_use"]
            print(f"turn {turn}: stop_reason={resp.stop_reason!r:<11} text={text_of(resp)[:40]!r:<44} tools={calls}")
        messages.append({"role": "assistant", "content": resp.content})   # keep the tool_use blocks
        if resp.stop_reason == "end_turn":
            return text_of(resp), messages
        if resp.stop_reason == "tool_use":
            results = [
                {"type": "tool_result", "tool_use_id": b.id, "content": json.dumps(impls[b.name](**b.input))}
                for b in resp.content if b.type == "tool_use"
            ]
            messages.append({"role": "user", "content": results})          # ALL results in one user turn
            continue
        raise RuntimeError(f"Unhandled stop_reason: {resp.stop_reason}")   # max_tokens, refusal, pause_turn, ...
    raise RuntimeError("Safety cap reached: investigate instead of returning a partial answer.")

In [ ]:
# A scripted fake client: replay responses to test loop control flow without the API (offline)
_ids = count(1)


def fake_msg(stop_reason: str, *blocks) -> Message:
    return Message(id=f"msg_{next(_ids)}", type="message", role="assistant", model="fake", content=list(blocks),
                   stop_reason=stop_reason, stop_sequence=None, usage=Usage(input_tokens=0, output_tokens=0))


def say(text: str) -> ApiTextBlock:
    return ApiTextBlock(type="text", text=text)


def call(name: str, **tool_input) -> ApiToolUseBlock:
    return ApiToolUseBlock(type="tool_use", id=f"toolu_{next(_ids)}", name=name, input=tool_input)


class FakeLLM:
    """Stands in for anthropic.Anthropic(): .messages.create() returns the next scripted Message."""

    def __init__(self, script: list[Message]):
        self._script = list(script)
        self.messages = SimpleNamespace(create=self._create)

    def _create(self, **_kwargs) -> Message:
        return self._script.pop(0)


def happy_script() -> list[Message]:
    return [
        fake_msg("tool_use", say("I'll verify the customer first."), call("get_customer", email="ana@example.com")),
        fake_msg("tool_use", say("Done verifying. Now the order."), call("lookup_order", order_id="A-1002")),
        fake_msg("end_turn", say("Order A-1002 was delivered. The total was $940.")),
    ]


answer, history = run_agent(FakeLLM(happy_script()), "Status of A-1002? I'm ana@example.com", TOOLS, IMPLS)
print("\nanswer:", answer, "| messages in history:", len(history))

> **Observe:** turn 2 says "Done…" **and** requests a tool. Only `stop_reason` tells the truth. The history alternates assistant (`tool_use`) and user (`tool_result`) turns. Drop those and the model loses what it learned.

In [ ]:
# (live) The same loop against the real API: ~3 small calls, < $0.01
require_live()
answer, history = run_agent(client, "I'm ana@example.com. What's the status of order A-1002?", TOOLS, IMPLS)
print("\n" + answer)

### Experiment: the three anti-patterns, side by side

The same scripted conversation is run under four stop rules. Only the `stop_reason` rule reaches the real answer. The others stop with a tool call still pending.

In [ ]:
# Stop-rule experiment (offline)
def run_with_stop_rule(llm, stop_rule, task: str = "Status of A-1002? I'm ana@example.com", cap: int = 10):
    messages = [{"role": "user", "content": task}]
    for turn in range(1, cap + 1):
        resp = llm.messages.create(model=MODEL, max_tokens=512, tools=TOOLS, messages=messages)
        messages.append({"role": "assistant", "content": resp.content})
        pending = [b.name for b in resp.content if b.type == "tool_use"]
        if stop_rule(resp, turn):
            return turn, text_of(resp), pending
        if pending:
            messages.append({"role": "user", "content": [
                {"type": "tool_result", "tool_use_id": b.id, "content": json.dumps(IMPLS[b.name](**b.input))}
                for b in resp.content if b.type == "tool_use"]})
    return cap, "(cap)", []


RULES = {
    "text contains 'done'  (anti-pattern)": lambda r, t: "done" in text_of(r).lower(),
    "response has text     (anti-pattern)": lambda r, t: any(b.type == "text" for b in r.content),
    "max_iterations=2      (anti-pattern)": lambda r, t: t >= 2,
    "stop_reason=end_turn  (correct)":      lambda r, t: r.stop_reason == "end_turn",
}
for name, rule in RULES.items():
    turn, final_text, pending = run_with_stop_rule(FakeLLM(happy_script()), rule)
    verdict = "OK" if not pending else f"BROKEN: pending {pending}"
    print(f"{name}  stopped at turn {turn}: {final_text[:38]!r:<42} {verdict}")

> **Try it:** add a fourth scripted response with `stop_reason="max_tokens"` before the final answer, and confirm that `run_agent` raises instead of returning a truncated answer.

### Same task, Agent SDK: custom tools via an in-process MCP server

With the Agent SDK you don't execute tools yourself. You define them with `@tool`, bundle them with `create_sdk_mcp_server`, and pass the server in `ClaudeAgentOptions(mcp_servers={...})`. Each tool is then named `mcp__<server key>__<tool name>`. Handlers return `{"content": [...]}`, plus `"is_error": True` to report a failure Claude should react to.

In [ ]:
# In-process MCP tools for a support desk (offline definition)
REFUNDS_ISSUED: list[dict] = []
ESCALATIONS: list[dict] = []


def as_result(data: Any, *, error: bool = False) -> dict[str, Any]:
    result = {"content": [{"type": "text", "text": json.dumps(data)}]}
    return {**result, "is_error": True} if error else result


@tool("get_customer", "Find a customer by email. Call this FIRST to verify identity.", {"email": str},
      annotations=ToolAnnotations(readOnlyHint=True))
async def get_customer_tool(args: dict[str, Any]) -> dict[str, Any]:
    return as_result(get_customer(args["email"]))   # "no_match" is a valid answer, not an error


@tool("lookup_order", "Get status and total (USD) for one order ID like A-1001.", {"order_id": str},
      annotations=ToolAnnotations(readOnlyHint=True))
async def lookup_order_tool(args: dict[str, Any]) -> dict[str, Any]:
    if not re.fullmatch(r"A-\d{4}", args["order_id"]):
        return as_result({"error": "invalid_order_id", "expected": "A-1234"}, error=True)
    return as_result(lookup_order(args["order_id"]))


@tool("process_refund", "Refund an order. Inputs: order_id, amount (USD), reason.",
      {"order_id": str, "amount": float, "reason": str})
async def process_refund_tool(args: dict[str, Any]) -> dict[str, Any]:
    REFUNDS_ISSUED.append(args)
    return as_result({"refund_id": f"RF-{len(REFUNDS_ISSUED):03d}", **args})


@tool("escalate_to_human", "Hand the case to a human agent. Inputs: customer_id, amount, reason.",
      {"customer_id": str, "amount": float, "reason": str})
async def escalate_to_human_tool(args: dict[str, Any]) -> dict[str, Any]:
    ESCALATIONS.append(args)
    return as_result({"ticket": f"T-{len(ESCALATIONS):03d}", "status": "queued_for_human"})


SUPPORT_SDK_TOOLS = [get_customer_tool, lookup_order_tool, process_refund_tool, escalate_to_human_tool]
support_server = create_sdk_mcp_server(name="support", version="1.0.0", tools=SUPPORT_SDK_TOOLS)
print("server config keys:", list(support_server))
print("tool names Claude sees:", [f"mcp__support__{t.name}" for t in SUPPORT_SDK_TOOLS])

In [ ]:
# Call the handlers directly: no model, no CLI (offline)
print(await lookup_order_tool.handler({"order_id": "A-1002"}))
print(await lookup_order_tool.handler({"order_id": "1002"}))          # is_error path
print(await get_customer_tool.handler({"email": "nobody@example.com"}))  # valid "no match"

> **Environment inspection:** every action tool here has a matching way to check its effect. `process_refund` returns a concrete `refund_id`, and `lookup_order` lets the agent re-read the order afterwards. An agent can only correct itself if it can observe the result of what it just did. [Theory](README.md#environment-inspection-let-the-agent-see-what-its-actions-did)

In [ ]:
# Helpers to drive and print an Agent SDK run (offline definitions; calling run_query is live)
def show(message) -> None:
    """One line per interesting event. '[sub]' marks messages produced inside a subagent."""
    pre = "    [sub] " if getattr(message, "parent_tool_use_id", None) else ""
    if isinstance(message, SystemMessage):
        print(f"{pre}system:{message.subtype}")
    elif isinstance(message, AssistantMessage):
        for b in message.content:
            if isinstance(b, TextBlock):
                print(f"{pre}assistant: {b.text[:110]!r}")
            elif isinstance(b, ToolUseBlock):
                print(f"{pre}tool_use -> {b.name} {json.dumps(b.input)[:110]}")
    elif isinstance(message, UserMessage) and isinstance(message.content, list):
        for b in message.content:
            if isinstance(b, ToolResultBlock):
                print(f"{pre}tool_result{' (ERROR)' if b.is_error else ''}: {str(b.content)[:110]}")
    elif isinstance(message, ResultMessage):
        cost = f"${message.total_cost_usd:.4f}" if message.total_cost_usd is not None else "n/a"
        print(f"RESULT subtype={message.subtype} turns={message.num_turns} cost={cost}")


async def run_query(prompt: str, options: ClaudeAgentOptions, *, verbose: bool = True):
    """Run query() to completion. Returns (ResultMessage | None, all messages)."""
    require_live()
    seen, result = [], None
    try:
        async for message in query(prompt=prompt, options=options):
            seen.append(message)
            if verbose:
                show(message)
            if isinstance(message, ResultMessage):
                result = message          # keep iterating: trailing system events can follow
    except Exception as err:              # single-shot query() raises after yielding an error result
        print(f"Run ended with an error: {err}")
    return result, seen


def base_options(**overrides) -> ClaudeAgentOptions:
    """Safe defaults for every live run in this notebook."""
    settings = dict(
        model=MODEL,
        cwd=SANDBOX_DIR,                 # a temp dir, not your repo
        setting_sources=[],              # ignore your personal ~/.claude settings, hooks and CLAUDE.md
        permission_mode="dontAsk",       # always explicit: unset may start in auto mode; dontAsk denies instead of prompting
        disallowed_tools=["Bash", "Write", "Edit", "NotebookEdit", "WebSearch", "WebFetch"],  # bare names: removed
        max_turns=12,                    # safety cap, not the success signal
        max_budget_usd=0.25,             # cost cap (subagent spend counts too)
    )
    settings.update(overrides)
    # Tool search is on by default: MCP tool schemas are deferred and loaded through the built-in ToolSearch tool.
    # ToolSearch needs no approval, so dontAsk doesn't block it; listing it here only documents the intent.
    settings["allowed_tools"] = [*settings.get("allowed_tools", []), "ToolSearch"]
    return ClaudeAgentOptions(**settings)


demo = base_options()
print({k: getattr(demo, k) for k in ("model", "permission_mode", "allowed_tools", "disallowed_tools",
                                      "max_turns", "max_budget_usd", "setting_sources")})

In [ ]:
# (live) The support task with query(): the SDK owns the loop. Capped at $0.25
support_opts = base_options(
    system_prompt="You are a concise support agent. Verify identity with get_customer before looking up orders.",
    mcp_servers={"support": support_server},
    allowed_tools=["mcp__support__get_customer", "mcp__support__lookup_order"],   # auto-approve just these
)
result, msgs = await run_query("I'm ana@example.com. What's the status of order A-1002?", support_opts)
if result and result.subtype == "success":
    print("\n" + result.result)

> **Observe:** you never wrote a loop. The stream shows `system:init`, then assistant `tool_use` messages, user `tool_result` messages, and finally `RESULT`. Tool search is on by default, so you may also see a `ToolSearch` call while Claude loads a deferred MCP tool schema. `process_refund` was never auto-approved, and `permission_mode="dontAsk"` means that a call to it would be denied, not prompted.

In [ ]:
# Handle every ResultMessage subtype explicitly (offline: fabricated results)
def handle_result(r: ResultMessage) -> str:
    match r.subtype:
        case "success":
            return f"done: {r.result}"
        case "error_max_turns":
            return f"hit max_turns after {r.num_turns} turns: investigate, or resume {r.session_id} with a higher cap"
        case "error_max_budget_usd":
            return "hit the cost cap: report partial progress; don't silently retry"
        case "error_during_execution":
            return f"interrupted: {r.errors}"
        case "error_max_structured_output_retries":
            return "no valid structured output: fall back or escalate"
        case _:
            return f"unknown subtype {r.subtype!r}: treat as failure"


for st in ["success", "error_max_turns", "error_max_budget_usd", "error_during_execution",
           "error_max_structured_output_retries"]:
    fake = ResultMessage(subtype=st, duration_ms=0, duration_api_ms=0, is_error=st != "success", num_turns=3,
                         session_id="sess-demo", result="Order A-1002 was delivered." if st == "success" else None)
    print(f"{st:38} -> {handle_result(fake)}")

> **Try it:** re-run the live cell with `max_turns=1` in `base_options(...)` and pass the `ResultMessage` to `handle_result`. You should get `error_max_turns`, and the `except` branch in `run_query` should print the error that `query()` raises after yielding it.

In [ ]:
# (live) Multi-turn in one process with ClaudeSDKClient: the second question reuses the session
require_live()
async with ClaudeSDKClient(options=support_opts) as sdk_client:
    await sdk_client.query("I'm ana@example.com. What's the status of order A-1001?")
    async for m in sdk_client.receive_response():
        show(m)
    await sdk_client.query("And my other order, A-1002?")        # no need to re-state the email
    async for m in sdk_client.receive_response():
        show(m)

## 3.2 `AgentDefinition` configuration

A subagent is an `AgentDefinition` in `ClaudeAgentOptions(agents={name: definition})`. The dict key is the name. `description` (routing signal) and `prompt` (its system prompt) are required. `tools` is the subagent's whole tool set, and omitting it means "inherit everything". Multi-word fields stay camelCase (`maxTurns`, `disallowedTools`), while `ClaudeAgentOptions` is snake_case. [Theory](README.md#32-agentdefinition-configuration)

In [ ]:
# What fields does the installed SDK accept? (offline)
for f in dataclasses.fields(AgentDefinition):
    required = f.default is dataclasses.MISSING and f.default_factory is dataclasses.MISSING
    print(f"{f.name:16} {'REQUIRED' if required else 'optional'}")

try:   # the community guide's pseudocode
    AgentDefinition(name="order-specialist", system_prompt="...", allowed_tools=["Read"])
except TypeError as err:
    print("\nGuide-style config fails:", err)

In [ ]:
# Two well-scoped support subagents (offline)
SUPPORT_AGENTS = {
    "order-specialist": AgentDefinition(
        description=("Verifies the customer by email and looks up order status and totals. "
                     "Use for any question about an existing order. Never issues refunds."),
        prompt=("You are an order specialist. Use only the data passed to you and your tools. "
                "Return JSON: {customer_id, order_id, status, total_usd}."),
        tools=["mcp__support__get_customer", "mcp__support__lookup_order"],
        model=SUBAGENT_MODEL,
        maxTurns=6,
    ),
    "refund-specialist": AgentDefinition(
        description=("Issues refunds for orders that the order specialist already verified, "
                     "or escalates to a human when policy blocks the refund."),
        prompt=("You process refunds. Re-check the order total before refunding. If a refund is denied, "
                "escalate with customer_id, amount and reason. Return JSON: {action, refund_id|ticket, amount}."),
        tools=["mcp__support__lookup_order", "mcp__support__process_refund", "mcp__support__escalate_to_human"],
        model=SUBAGENT_MODEL,
        maxTurns=6,
    ),
}
print(json.dumps({k: {f: v for f, v in dataclasses.asdict(a).items() if v is not None}
                  for k, a in SUPPORT_AGENTS.items()}, indent=2)[:900])

In [ ]:
# A small linter for agent definitions (offline)
def _words(text: str) -> set[str]:
    return {w for w in re.findall(r"[a-z]+", text.lower()) if len(w) > 3}


def audit_agents(agents: dict[str, AgentDefinition], max_tools: int = 5) -> list[str]:
    issues = []
    for name, a in agents.items():
        if a.tools is None:
            issues.append(f"{name}: no `tools` -> inherits EVERY tool available to subagents")
        elif len(a.tools) > max_tools:
            issues.append(f"{name}: {len(a.tools)} tools -> tool-selection errors rise; split the role")
        if set(a.tools or []) & {"Agent", "Task"}:
            issues.append(f"{name}: can spawn its own subagents (intended?)")
        if len(a.description.split()) < 8:
            issues.append(f"{name}: description too short to route on")
    names = list(agents)
    for i, x in enumerate(names):
        for y in names[i + 1:]:
            wx, wy = _words(agents[x].description), _words(agents[y].description)
            overlap = len(wx & wy) / max(1, len(wx | wy))
            if overlap > 0.5:
                issues.append(f"{x} / {y}: descriptions overlap {overlap:.0%} -> misrouting risk")
    return issues


BAD_AGENTS = {
    "helper-1": AgentDefinition(description="Helps with customer questions.", prompt="Help the customer."),
    "helper-2": AgentDefinition(description="Helps with customer order questions.", prompt="Help.",
                                tools=["Read", "Grep", "Glob", "Bash", "Edit", "Write", "WebSearch"]),
}
print("BAD:", *audit_agents(BAD_AGENTS), sep="\n  ")
print("GOOD:", audit_agents(SUPPORT_AGENTS) or "no issues")

### `allowed_tools` vs `tools` vs `disallowed_tools`

`allowed_tools` only **auto-approves**, so unlisted tools still exist and fall through to the permission mode. A bare name in `disallowed_tools` **removes** a tool. `AgentDefinition.tools` **is** the subagent's tool set. The function below is a simplified, offline model of the documented evaluation order (hooks, then deny rules, permission mode, allow rules, `can_use_tool`). It's a teaching aid and leaves out `ask` rules, `acceptEdits`, `plan`, and `auto` mode.

In [ ]:
# Who can actually run what? (offline teaching model, not the SDK)
from fnmatch import fnmatch

NEEDS_NO_APPROVAL = {"Agent", "ToolSearch", "Read", "Glob", "Grep"}   # e.g. Agent, ToolSearch, reads inside the working dir


def explain_tool(tool_name: str, opts: ClaudeAgentOptions) -> str:
    if tool_name in opts.disallowed_tools:
        return "REMOVED from context (bare-name disallowed_tools)"
    if opts.permission_mode == "bypassPermissions":
        return "APPROVED by bypassPermissions (allowed_tools does not constrain this mode)"
    if any(tool_name == rule or (rule.startswith("mcp__") and fnmatch(tool_name, rule)) for rule in opts.allowed_tools):
        return "approved (allowed_tools)"
    if tool_name in NEEDS_NO_APPROVAL:
        return "runs (needs no approval)"
    if opts.permission_mode == "dontAsk":
        return "DENIED (dontAsk never prompts)"
    return "falls through to can_use_tool -> denied if you provided none"


CONFIGS = {
    "allowed=[Read] + bypass": ClaudeAgentOptions(allowed_tools=["Read"], permission_mode="bypassPermissions"),
    "allowed=[Read] + dontAsk": ClaudeAgentOptions(allowed_tools=["Read", "mcp__support__*"], permission_mode="dontAsk"),
    "bypass + disallowed=[Bash]": ClaudeAgentOptions(permission_mode="bypassPermissions", disallowed_tools=["Bash"]),
}
for label, opts in CONFIGS.items():
    print(label)
    for t in ["Read", "Bash", "Agent", "mcp__support__process_refund"]:
        print(f"   {t:30} {explain_tool(t, opts)}")

> **Observe:** with `bypassPermissions`, `allowed_tools=["Read"]` still lets `Bash` run. That is the classic "allowed_tools is a sandbox" mistake. To restrict, remove tools (`disallowed_tools`, `AgentDefinition.tools`), add hooks, or pair `allowed_tools` with `dontAsk`. Hooks run before all of this, and a hook `deny` holds even under `bypassPermissions`.

## 3.3 Hub-and-spoke: coordinator and subagents

The coordinator decomposes the goal, picks which subagents to use, writes a self-contained prompt for each, merges and validates results, handles errors, and is the single voice to the user. Subagents start with a fresh context and return only their final message. When every subagent "succeeds" but the report has gaps, check the coordinator's decomposition first. [Theory](README.md#33-hub-and-spoke-coordinator-and-subagents)

In [ ]:
# A mock research corpus served by an in-process MCP server (offline definition)
CORPUS = [
    {"id": "D1", "area": "visual art", "title": "Galleries test AI-assisted generative installations",
     "published": 1767225600, "url": "https://example.com/d1",
     "text": "Three European galleries showed generative installations; curators report mixed audience reactions."},
    {"id": "D2", "area": "graphic design", "title": "Design agencies report faster concept rounds with AI",
     "published": "2026-02-14", "url": "https://example.com/d2",
     "text": "Agencies cut first-concept time by about 40 percent but still hand-finish final brand assets."},
    {"id": "D3", "area": "photography", "title": "Stock photo libraries add labels for AI-generated images",
     "published": "Mar 5, 2026", "url": "https://example.com/d3",
     "text": "Major stock libraries now require provenance labels; contributor payouts for generic shots fell."},
    {"id": "D4", "area": "music", "title": "Labels negotiate licensing for AI-trained music models",
     "published": "2026-04-02T09:30:00Z", "url": "https://example.com/d4",
     "text": "Record labels signed licensing deals covering training data; independent artists want opt-outs."},
    {"id": "D5", "area": "film", "title": "Studios use AI for previsualization and dubbing",
     "published": 1772668800, "url": "https://example.com/d5",
     "text": "Studios use AI previs to plan shots and AI dubbing for secondary markets; voice actors negotiate consent."},
    {"id": "D6", "area": "writing", "title": "Publishers set disclosure rules for AI-assisted books",
     "published": "Jan 20, 2026", "url": "https://example.com/d6",
     "text": "Publishers require authors to disclose AI assistance; translation workflows changed fastest."},
    {"id": "D7", "area": "games", "title": "Game studios adopt AI for asset generation as unions push back",
     "published": "2026-05-11", "url": "https://example.com/d7",
     "text": "Studios generate background assets with AI; unions seek credit and consent rules for performers."},
    {"id": "D8", "area": "music", "title": "Royalty disputes over AI voice clones",
     "published": "Jun 1, 2026", "url": "https://example.com/d8",
     "text": "Voice-clone tracks triggered takedowns; collecting societies are drafting royalty rules."},
]


@tool("search_corpus", "Search the research corpus by keywords. Returns up to 3 documents "
      "(id, area, title, published, url). An empty list means no match, which is a valid finding.",
      {"query": str}, annotations=ToolAnnotations(readOnlyHint=True))
async def search_corpus(args: dict[str, Any]) -> dict[str, Any]:
    terms = set(re.findall(r"[a-z]+", args["query"].lower()))
    scored = []
    for doc in CORPUS:
        words = set(re.findall(r"[a-z]+", f"{doc['area']} {doc['title']} {doc['text']}".lower()))
        if hits := len(terms & words):
            scored.append((hits, {k: doc[k] for k in ("id", "area", "title", "published", "url")}))
    top = [d for _, d in sorted(scored, key=lambda s: -s[0])[:3]]
    return as_result({"query": args["query"], "results": top})


@tool("get_document", "Get the full text of one corpus document by id (e.g. D4).", {"doc_id": str},
      annotations=ToolAnnotations(readOnlyHint=True))
async def get_document(args: dict[str, Any]) -> dict[str, Any]:
    doc = next((d for d in CORPUS if d["id"] == args["doc_id"]), None)
    if doc is None:
        return as_result({"error": "unknown_doc_id", "doc_id": args["doc_id"]}, error=True)
    return as_result(doc)


research_server = create_sdk_mcp_server(name="research", version="1.0.0", tools=[search_corpus, get_document])
print(await search_corpus.handler({"query": "music licensing"}))

In [ ]:
# Decomposition coverage: the "AI in creative industries" failure from the README (offline)
REQUIRED_AREAS = {
    "visual art": ["art", "visual"], "graphic design": ["design"], "photography": ["photo"],
    "music": ["music"], "film": ["film", "movie"], "writing": ["writing", "publishing", "books"],
    "games": ["game"],
}


def coverage(subtopics: list[str]) -> tuple[set[str], set[str]]:
    text = " ".join(subtopics).lower()
    covered = {area for area, kws in REQUIRED_AREAS.items() if any(k in text for k in kws)}
    return covered, set(REQUIRED_AREAS) - covered


NARROW_PLAN = ["digital art", "graphic design", "photography"]
for label, plan in [("narrow", NARROW_PLAN), ("full", NARROW_PLAN + ["music", "film & TV", "writing & publishing", "games"])]:
    covered, missing = coverage(plan)
    print(f"{label:7} covered={sorted(covered)}\n        missing={sorted(missing) or '-'}")

In [ ]:
# Two coordinator prompts and three subagents for the live comparison (offline definitions)
NARROW_COORDINATOR = (
    "You coordinate research. Split the topic into exactly these subtopics: digital art, graphic design, "
    "photography. Delegate each to corpus-researcher, then pass all findings to the synthesizer."
)
COVERAGE_COORDINATOR = (
    "You coordinate research. Goal: complete coverage with cited claims.\n"
    "- Partition the topic into NON-OVERLAPPING subtopics that together cover ALL major areas.\n"
    "- Spawn one corpus-researcher per subtopic, in parallel (several Agent calls in one response).\n"
    "- Pass every finding verbatim to the synthesizer.\n"
    "- Before answering, list any area with no findings as a gap; re-delegate once if needed."
)
RESEARCH_AGENTS = {
    "corpus-researcher": AgentDefinition(
        description="Searches the research corpus for ONE assigned subtopic and returns cited findings.",
        prompt=("Research only the subtopic you are given. Return a JSON list of "
                "{claim, doc_id, url, published}. If nothing matches, return [] and say so."),
        tools=["mcp__research__search_corpus", "mcp__research__get_document"],
        model=SUBAGENT_MODEL, maxTurns=5,
    ),
    "doc-analyst": AgentDefinition(
        description="Reads one or more corpus documents by id and extracts risks and open questions.",
        prompt="Read the given doc ids. Return JSON {doc_id, risks: [...], open_questions: [...]}.",
        tools=["mcp__research__get_document"],
        model=SUBAGENT_MODEL, maxTurns=4,
    ),
    "synthesizer": AgentDefinition(
        description="Merges research findings passed in its prompt into a short cited report.",
        prompt=("Write a report using ONLY the findings provided. Keep every claim's doc_id. "
                "Mark conflicting values instead of choosing one."),
        tools=["Read"],
        model=SUBAGENT_MODEL, maxTurns=3,
    ),
}
print(audit_agents(RESEARCH_AGENTS) or "agents OK")

In [ ]:
# Structured error propagation: "0 results" is a finding, a timeout is a failure (offline)
def coordinator_decision(report: dict) -> str:
    if report["status"] == "ok" and not report["results"]:
        return "accept: record 'no sources found' as a gap in the report"
    if report["status"] == "ok":
        return f"accept {len(report['results'])} findings"
    if report.get("failure_type") == "timeout":
        return f"retry once or reroute to {report['alternatives'][0]}; keep {len(report['partial_results'])} partial results"
    return "escalate: unknown failure"


reports = [
    {"status": "ok", "query": "AI dubbing regulation Brazil", "results": []},
    {"status": "error", "failure_type": "timeout", "attempted": "news API: 'AI dubbing'",
     "partial_results": [{"claim": "Studios use AI dubbing", "doc_id": "D5"}], "alternatives": ["corpus search"]},
    {"status": "error", "message": "search unavailable"},   # generic: the coordinator can't act on this
]
for r in reports:
    print(f"{json.dumps(r)[:70]:72} -> {coordinator_decision(r)}")

### Dynamic selection and the iterative refinement loop

Two coordinator skills from the README, modelled offline. First, **dynamic selection**: a narrow lookup needs one specialist, and only a broad question needs the full research-then-synthesis pipeline. Second, the **refinement loop**: synthesize, compare the draft with the quality criteria (here, required areas), re-delegate **targeted** queries for the gaps only, and re-synthesize. The round limit is a guardrail. Any gap left at the end is reported, not hidden. The tool handler stands in for a researcher subagent, so this runs without the model.

In [ ]:
# Dynamic subagent selection + an iterative refinement loop (offline: the tool handler stands in for a subagent)
def plan_delegation(request: str) -> list[str]:
    """Pick only the subagents this request needs instead of always running the whole pipeline."""
    text = request.lower()
    broad = any(w in text for w in ("research", "impact", "landscape", "overview"))
    if re.search(r"\bd\d+\b", text) and not broad:
        return ["doc-analyst"]                        # narrow: one known document, no synthesis needed
    return ["corpus-researcher", "synthesizer"] if broad else ["corpus-researcher"]


for request in ["Summarize the risks in D4", "Research: the impact of AI on the creative industries",
                "Any news on AI dubbing?"]:
    print(f"{request!r:58} -> {plan_delegation(request)}")


def result_text(result: dict) -> str:
    return "".join(b.get("text", "") for b in result["content"])


async def research_area(area: str) -> list[dict]:
    """Stand-in for ONE corpus-researcher call: cited findings for exactly one area."""
    hits = json.loads(result_text(await search_corpus.handler({"query": area})))["results"]
    return [{"claim": h["title"], "doc_id": h["id"], "area": h["area"]} for h in hits if h["area"] == area]


def synthesize(findings: list[dict]) -> dict:
    return {"sections": sorted({f["area"] for f in findings}), "claims": findings}


async def refine(first_plan: list[str], required: list[str], max_rounds: int = 3) -> tuple[dict, list[str]]:
    tried = set(first_plan)
    findings = [f for area in first_plan for f in await research_area(area)]
    report = synthesize(findings)
    for round_no in range(1, max_rounds + 1):
        missing = sorted(set(required) - set(report["sections"]))
        retry = [a for a in missing if a not in tried]          # don't re-ask what already came back empty
        print(f"round {round_no}: {len(report['sections'])} sections | gaps={missing or '-'} | re-delegate={retry or '-'}")
        if not retry:
            return report, missing                               # done, or nothing new to try: report what's left
        for area in retry:                                       # targeted queries for the gaps only
            tried.add(area)
            findings += await research_area(area)
        report = synthesize(findings)                            # re-run synthesis with the new findings
    return report, sorted(set(required) - set(report["sections"]))


REQUIRED = [*REQUIRED_AREAS, "architecture"]                     # the corpus has nothing on architecture
report, remaining = await refine(["visual art", "graphic design", "photography"], REQUIRED)
print("final sections:", report["sections"], "\nreported gaps:", remaining)

In [ ]:
# (live) Narrow vs coverage-driven coordinator on the same corpus. Two runs, capped at $0.30 each
require_live()


def spawned_prompts(messages) -> list[str]:
    return [b.input.get("prompt", "") for m in messages if isinstance(m, AssistantMessage)
            for b in m.content if isinstance(b, ToolUseBlock) and b.name in ("Agent", "Task")]


runs = {}
for label, coordinator in [("narrow", NARROW_COORDINATOR), ("coverage", COVERAGE_COORDINATOR)]:
    opts = base_options(
        system_prompt=coordinator, agents=RESEARCH_AGENTS,
        mcp_servers={"research": research_server},
        allowed_tools=["Agent", "mcp__research__*"],
        max_turns=15, max_budget_usd=0.30,
        env={"CLAUDE_CODE_MAX_SUBAGENT_SPAWN_DEPTH": "1"},   # subagents can't spawn more subagents
    )
    print(f"\n=== {label} ===")
    runs[label] = await run_query("Research: the impact of AI on the creative industries.", opts, verbose=False)
    covered, missing = coverage(spawned_prompts(runs[label][1]))
    print(f"delegated areas: {sorted(covered)} | missing: {sorted(missing) or '-'}")

> **Observe:** the narrow run may finish with `success` and still miss music, film, writing, and games. The subagents did exactly what they were asked. The coverage prompt states goals and quality criteria rather than a fixed script, and that is what closes the gap.
>
> **Try it:** add a check to `COVERAGE_COORDINATOR` that requires a `doc-analyst` pass on the two most-cited documents, then see whether the coordinator runs it sequentially after research, since it depends on research results.

## 3.4 The subagent tool (`Agent`, formerly `Task`)

The coordinator spawns a subagent by calling the built-in **`Agent`** tool with `subagent_type` (the agent's name; omitted means the built-in `general-purpose` agent) and a `prompt`, which is the **only** context the subagent receives. `tool_use` blocks say `"Agent"`, while the `system:init` tool list still says `"Task"`, so match both. Several `Agent` calls in one response run in parallel. [Theory](README.md#34-the-subagent-tool-agent-formerly-task)

| `Agent` tool input | Meaning |
|---|---|
| `description` | Short label for the task |
| `prompt` | Everything the subagent knows about the task, so make it self-contained |
| `subagent_type` | Key from `agents={...}`; omitted means `general-purpose` |
| `model` | Optional per-call override (`sonnet`, `opus`, `haiku`, `fable`) |
| `run_in_background` | Subagents run in the background by default; Claude sets `false` when it needs the result first |

In [ ]:
# Weak vs strong delegation prompt (offline)
def build_delegation_prompt(task: str, documents: dict[str, str], prior_findings: list[dict], schema: str) -> str:
    docs = "\n".join(f'<document id="{k}">{v}</document>' for k, v in documents.items())
    findings = "\n".join(
        f'<finding source="{f["source"]}" retrieved="{f["retrieved"]}">{f["claim"]}</finding>' for f in prior_findings)
    return (f"{task}\n<documents>\n{docs}\n</documents>\n<prior_findings>\n{findings}\n</prior_findings>\n"
            f"Return JSON: {schema}. If a field is unknown, use null.")


weak = "Analyze the document."
strong = build_delegation_prompt(
    "Find licensing risks for AI-generated music in the documents below.",
    {"D4": CORPUS[3]["text"], "D8": CORPUS[7]["text"]},
    [{"source": "corpus-researcher", "retrieved": "2026-10-01", "claim": "Labels signed training-data deals (D4)"}],
    "[{risk, doc_id, quote, severity}]",
)
print("WEAK:\n", weak, "\n\nSTRONG:\n", strong)

In [ ]:
# Trace a run: who spawned what, in parallel or not, and which messages came from inside subagents (offline)
def trace(messages, *, verbose: bool = True) -> dict:
    spawns: dict[str, str] = {}          # Agent tool_use id -> subagent_type
    per_response: dict[str, list[str]] = {}
    inside = 0
    for m in messages:
        if isinstance(m, AssistantMessage):
            for b in m.content:
                if isinstance(b, ToolUseBlock) and b.name in ("Agent", "Task"):   # match both names
                    kind = b.input.get("subagent_type") or "general-purpose"
                    spawns[b.id] = kind
                    if m.message_id:
                        per_response.setdefault(m.message_id, []).append(kind)
                    if verbose:
                        print(f"spawn  {kind:18} via {b.name:5} id={b.id}")
        if getattr(m, "parent_tool_use_id", None):
            inside += 1
            if verbose:
                print(f"   ...{type(m).__name__} inside {spawns.get(m.parent_tool_use_id, '?')}")
        if isinstance(m, ResultMessage) and verbose:
            print("result:", m.subtype)
    return {"spawned": list(spawns.values()),
            "parallel_batches": [v for v in per_response.values() if len(v) > 1],
            "messages_inside_subagents": inside}


# A fabricated run, built from the SDK's own message classes
FAKE_RUN = [
    SystemMessage(subtype="init", data={"tools": ["Task", "mcp__research__search_corpus"]}),
    AssistantMessage(model="fake", message_id="msg_1", content=[ToolUseBlock(
        id="tu_music", name="Agent", input={"subagent_type": "corpus-researcher", "description": "music",
                                            "prompt": "Research ONLY music. Return JSON findings."})]),
    AssistantMessage(model="fake", message_id="msg_1", content=[ToolUseBlock(   # same response -> parallel
        id="tu_film", name="Agent", input={"subagent_type": "corpus-researcher", "description": "film",
                                           "prompt": "Research ONLY film. Return JSON findings."})]),
    AssistantMessage(model="fake", parent_tool_use_id="tu_music", content=[ToolUseBlock(
        id="tu_m1", name="mcp__research__search_corpus", input={"query": "music"})]),
    AssistantMessage(model="fake", parent_tool_use_id="tu_film", content=[ToolUseBlock(
        id="tu_f1", name="mcp__research__search_corpus", input={"query": "film"})]),
    UserMessage(content=[ToolResultBlock(tool_use_id="tu_music", content='[{"claim": "...", "doc_id": "D4"}]')]),
    UserMessage(content=[ToolResultBlock(tool_use_id="tu_film", content='[{"claim": "...", "doc_id": "D5"}]')]),
    AssistantMessage(model="fake", message_id="msg_2", content=[ToolUseBlock(
        id="tu_syn", name="Task", input={"subagent_type": "synthesizer", "description": "merge",   # older CLI name
                                         "prompt": "<findings>...</findings> Write the report."})]),
    ResultMessage(subtype="success", duration_ms=1, duration_api_ms=1, is_error=False, num_turns=4,
                  session_id="sess-demo", result="Report ..."),
]
print(trace(FAKE_RUN))

In [ ]:
# (live) Explicit invocation + tracing. Capped at $0.25
opts = base_options(
    system_prompt="You coordinate research. Delegate fact-finding; never search yourself.",
    agents=RESEARCH_AGENTS, mcp_servers={"research": research_server},
    allowed_tools=["Agent", "mcp__research__*"],
    env={"CLAUDE_CODE_MAX_SUBAGENT_SPAWN_DEPTH": "1", "CLAUDE_CODE_MAX_CONCURRENT_SUBAGENTS": "3"},
)
result, msgs = await run_query(
    "Use the corpus-researcher agent twice, in parallel: once for music, once for film. Then summarize in 3 bullets.",
    opts, verbose=False)
print(trace(msgs))
if result and result.subtype == "success":
    print(result.result)

> **Observe:** messages with `parent_tool_use_id` come from inside a subagent. Their intermediate tool calls never enter the coordinator's context, because only the final report comes back. That is why subagents also help with context management.
>
> **Try it:** delegate `"Analyze the document."` (the weak prompt) to `doc-analyst` and compare its output with a run that uses `strong`.

## 3.5 Hooks in the Agent SDK

Hooks are async callbacks that run in **your** process at fixed lifecycle points. Register them with `hooks={"PreToolUse": [HookMatcher(matcher=..., hooks=[cb])]}`. The callback signature is `cb(input_data, tool_use_id, context) -> dict`, and returning `{}` lets the call through. A `PreToolUse` deny goes in `hookSpecificOutput.permissionDecision`, and Claude reads your `permissionDecisionReason`. Matchers only see the **tool name**, so inspect arguments inside the callback. [Theory](README.md#35-hooks-in-the-agent-sdk)

In [ ]:
# Policy hooks for the support desk (offline definitions)
REFUND_LIMIT_USD = 500
verified_sessions: set[str] = set()
AUDIT_LOG: list[str] = []


def deny(reason: str) -> dict:
    return {"hookSpecificOutput": {"hookEventName": "PreToolUse",
                                   "permissionDecision": "deny", "permissionDecisionReason": reason}}


def tool_text(tool_response: Any) -> str:
    """MCP results arrive as content blocks (a list, or a dict with 'content'). Join their text."""
    blocks = tool_response.get("content", []) if isinstance(tool_response, dict) else tool_response
    if isinstance(blocks, str):
        return blocks
    return "".join(b.get("text", "") for b in blocks or [] if isinstance(b, dict))


async def enforce_refund_limit(input_data, tool_use_id, context):        # PreToolUse
    try:
        amount = float(input_data["tool_input"].get("amount"))
    except (TypeError, ValueError):
        return deny("Refund amount is missing or not a number. Look up the order total first.")  # fail closed
    if amount > REFUND_LIMIT_USD:
        return deny(f"Refunds over ${REFUND_LIMIT_USD} need human approval. Call "
                    "mcp__support__escalate_to_human with customer_id, amount and reason.")
    return {}


async def require_verified_customer(input_data, tool_use_id, context):   # PreToolUse precondition
    if input_data["session_id"] not in verified_sessions:
        return deny("Call mcp__support__get_customer first to verify the customer's identity.")
    return {}


async def mark_verified(input_data, tool_use_id, context):               # PostToolUse on get_customer
    try:
        data = json.loads(tool_text(input_data["tool_response"]) or "{}")
    except json.JSONDecodeError:
        return {}
    if "customer_id" in data:              # a successful call can still be "no_match"
        verified_sessions.add(input_data["session_id"])
    return {}


async def log_subagent(input_data, tool_use_id, context):                # SubagentStart / SubagentStop
    AUDIT_LOG.append(f"{input_data['hook_event_name']}: {input_data.get('agent_type')} ({input_data.get('agent_id')})")
    return {}


SUPPORT_HOOKS = {
    "PreToolUse": [
        HookMatcher(matcher="mcp__support__lookup_order|mcp__support__process_refund",
                    hooks=[require_verified_customer]),
        HookMatcher(matcher="mcp__support__process_refund", hooks=[enforce_refund_limit]),
    ],
    "PostToolUse": [HookMatcher(matcher="mcp__support__get_customer", hooks=[mark_verified])],
    "SubagentStart": [HookMatcher(hooks=[log_subagent])],
    "SubagentStop": [HookMatcher(hooks=[log_subagent])],
}
print({event: len(matchers) for event, matchers in SUPPORT_HOOKS.items()})

In [ ]:
# A PreToolUse simulator: matcher rules + "most restrictive decision wins" (offline teaching model)
PRECEDENCE = {"allow": 0, "ask": 1, "defer": 2, "deny": 3}
EXACT_MATCHER = re.compile(r"^[A-Za-z0-9_\-, |]+$")


def matcher_matches(matcher: str | None, tool_name: str) -> bool:
    if matcher in (None, "", "*"):
        return True
    if EXACT_MATCHER.match(matcher):                       # exact name, or a list split by | or ,
        return tool_name in {p.strip() for p in re.split(r"[|,]", matcher)}
    return re.search(matcher, tool_name) is not None       # otherwise: unanchored regex


async def simulate_pre_tool_use(hooks: dict, tool_name: str, tool_input: dict, session_id: str = "sess-demo"):
    payload = {"hook_event_name": "PreToolUse", "session_id": session_id, "transcript_path": "", "cwd": SANDBOX_DIR,
               "tool_name": tool_name, "tool_input": tool_input, "tool_use_id": "toolu_sim"}
    callbacks = [cb for m in hooks.get("PreToolUse", []) if matcher_matches(m.matcher, tool_name) for cb in m.hooks]
    outputs = await asyncio.gather(*(cb(payload, "toolu_sim", {"signal": None}) for cb in callbacks))  # run in parallel
    decision, reason = "no decision -> normal permission flow", ""
    best = -1
    for out in outputs:
        hso = out.get("hookSpecificOutput", {})
        if (d := hso.get("permissionDecision")) and PRECEDENCE[d] > best:
            best, decision, reason = PRECEDENCE[d], d, hso.get("permissionDecisionReason", "")
    return decision, reason


async def simulate_post_tool_use(hooks: dict, tool_name: str, tool_response: Any, session_id: str = "sess-demo"):
    payload = {"hook_event_name": "PostToolUse", "session_id": session_id, "transcript_path": "", "cwd": SANDBOX_DIR,
               "tool_name": tool_name, "tool_input": {}, "tool_response": tool_response, "tool_use_id": "toolu_sim"}
    callbacks = [cb for m in hooks.get("PostToolUse", []) if matcher_matches(m.matcher, tool_name) for cb in m.hooks]
    return await asyncio.gather(*(cb(payload, "toolu_sim", {"signal": None}) for cb in callbacks))


verified_sessions.clear()
print("1", await simulate_pre_tool_use(SUPPORT_HOOKS, "mcp__support__process_refund", {"amount": 50}))
await simulate_post_tool_use(SUPPORT_HOOKS, "mcp__support__get_customer",
                             {"content": [{"type": "text", "text": json.dumps(CUSTOMERS["ana@example.com"])}]})
print("2", await simulate_pre_tool_use(SUPPORT_HOOKS, "mcp__support__process_refund", {"amount": 200}))
print("3", await simulate_pre_tool_use(SUPPORT_HOOKS, "mcp__support__process_refund", {"amount": "940.00"}))
print("4", await simulate_pre_tool_use(SUPPORT_HOOKS, "mcp__support__lookup_order", {"order_id": "A-1002"}))

In [ ]:
# Matcher semantics at a glance (offline)
for matcher, name in [("Edit|Write", "Write"), ("Edit, Write", "Edit"), ("Edit|Write", "NotebookEdit"),
                      ("Edit.*", "NotebookEdit"), ("^Edit$", "NotebookEdit"), ("^mcp__", "mcp__support__lookup_order"),
                      ("mcp__support__.*", "mcp__billing__refund"), ("process_refund", "mcp__support__process_refund")]:
    print(f"{matcher!r:22} vs {name:30} -> {matcher_matches(matcher, name)}")

> **Observe:** `"process_refund"` does **not** match `mcp__support__process_refund`, because exact matchers need the full `mcp__<server>__<tool>` name. `"Edit.*"` is an unanchored regex and also matches `NotebookEdit`.

### `PostToolUse` normalization

Third-party tools return epoch seconds, ISO strings, `"Mar 5, 2026"`, and numeric status codes. One `PostToolUse` hook can rewrite every result into one format with `updatedToolOutput` before Claude reads it. The value must keep the **same shape** as the `tool_response` it replaces. Built-in tools return structured objects (`Bash`: `{stdout, stderr, interrupted, isImage}`), and a mismatched replacement for them is ignored. MCP output is passed through without validation, so a wrong shape or stripped error details reach Claude. The hook below accepts an MCP result object (`{"content": [...]}`) or a bare block list, and returns whichever shape it received. The docs don't pin down the exact MCP shape the SDK delivers, so print one raw `tool_response` in a live run before you simplify the hook.

In [ ]:
# Normalizer + PostToolUse hook (offline)
STATUS_CODES = {0: "pending", 1: "processing", 2: "shipped", 3: "delivered"}
DATE_KEYS = {"published", "eta", "created_at", "date"}


def to_iso_date(value: Any) -> Any:
    if isinstance(value, bool) or value is None:
        return value
    if isinstance(value, (int, float)):
        return datetime.fromtimestamp(value, tz=timezone.utc).date().isoformat()
    if isinstance(value, str):
        try:
            return datetime.fromisoformat(value).date().isoformat()
        except ValueError:
            pass
        for fmt in ("%b %d, %Y", "%d/%m/%Y"):
            try:
                return datetime.strptime(value, fmt).date().isoformat()
            except ValueError:
                pass
    return value          # unknown format: leave it rather than guess


def normalize(data: Any) -> Any:
    if isinstance(data, list):
        return [normalize(x) for x in data]
    if isinstance(data, dict):
        out = {}
        for k, v in data.items():
            if k in DATE_KEYS:
                out[k] = to_iso_date(v)
            elif k == "status" and isinstance(v, int):
                out[k] = STATUS_CODES.get(v, f"unknown({v})")
            else:
                out[k] = normalize(v)
        return out
    return data


async def normalize_tool_output(input_data, tool_use_id, context):     # PostToolUse
    raw = input_data["tool_response"]
    try:
        data = json.loads(tool_text(raw))
    except (json.JSONDecodeError, TypeError):
        return {}                                                       # not JSON: leave untouched
    blocks = [{"type": "text", "text": json.dumps(normalize(data))}]
    new_output = {**raw, "content": blocks} if isinstance(raw, dict) else blocks   # keep the shape we received
    return {"hookSpecificOutput": {"hookEventName": "PostToolUse", "updatedToolOutput": new_output}}


carrier_response = {"content": [{"type": "text", "text": json.dumps(
    {"shipments": [{"id": "S1", "status": 2, "eta": 1772668800}, {"id": "S2", "status": 3, "eta": "Mar 5, 2026"}]})}]}
NORMALIZE_HOOKS = {"PostToolUse": [HookMatcher(matcher="^mcp__", hooks=[normalize_tool_output])]}
[out] = await simulate_post_tool_use(NORMALIZE_HOOKS, "mcp__carrier__track", carrier_response)
new_output = out["hookSpecificOutput"]["updatedToolOutput"]
print("same shape as the input:", type(new_output) is type(carrier_response), "| keys:", list(new_output))
print(tool_text(new_output))
[out_list] = await simulate_post_tool_use(NORMALIZE_HOOKS, "mcp__carrier__track", carrier_response["content"])
print("bare block list in -> list out:", isinstance(out_list["hookSpecificOutput"]["updatedToolOutput"], list))

In [ ]:
# Experiment: "skipped verification" rate, prompt-only vs hook (offline toy model)
def violation_rate(n_runs: int, model_skip_rate: float, hook: bool, seed: int = 7) -> float:
    """Each run, the model 'forgets' to verify with probability model_skip_rate. A hook blocks every such call."""
    rng = random.Random(seed)
    skipped = sum(rng.random() < model_skip_rate for _ in range(n_runs))
    return 0.0 if hook else skipped / n_runs


for label, p, hook in [("prompt only", 0.09, False), ("better prompt", 0.04, False), ("PreToolUse hook", 0.04, True)]:
    print(f"{label:16} -> refunds before verification: {violation_rate(10_000, p, hook):.2%}")

> **Observe:** prompt work lowers the rate, and only code removes it. Apply the cost-of-failure test from [Hooks vs prompt instructions](README.md#hooks-vs-prompt-instructions): if a single violation has financial, legal, or safety impact, enforce the rule in a hook.

In [ ]:
# (live) The support agent with hooks: verification is enforced in code, not mentioned in the prompt. Capped at $0.25
require_live()
verified_sessions.clear(); REFUNDS_ISSUED.clear(); ESCALATIONS.clear()
hooked_opts = base_options(
    system_prompt="You are a support agent. Resolve the customer's request with the tools available.",
    mcp_servers={"support": support_server},
    allowed_tools=["mcp__support__*"],      # glob is allowed after a literal mcp__<server>__ prefix
    hooks=SUPPORT_HOOKS,
)
result, msgs = await run_query(
    "I'm ana@example.com. Order A-1002 arrived broken. Refund the full amount please.", hooked_opts)
print("\nrefunds issued:", REFUNDS_ISSUED)
print("escalations:", ESCALATIONS)

> **Observe:** look for the `tool_result (ERROR)` line carrying your deny reason. Claude adapts and calls `escalate_to_human`, and `REFUNDS_ISSUED` stays empty for the $940 order.
>
> **Try it:** lower `REFUND_LIMIT_USD` to `100`, ask for a refund on `A-1001` ($129), and confirm the same path. Then add a `"PreToolUse"` hook with no matcher that appends every `tool_name` to `AUDIT_LOG`.

## 3.6 Sessions: resume and fork

Every run has a `session_id` (from `ResultMessage.session_id`). `resume=session_id` continues it. `resume=..., fork_session=True` branches into a **new** session ID and leaves the original untouched. `continue_conversation=True` picks up the most recent session. When most earlier tool results are stale, start fresh with a structured summary instead of resuming. [Theory](README.md#36-sessions-resume-and-fork-bonus-from-domain-17)

In [ ]:
# Session option shapes (offline)
session_id = "sess-from-an-earlier-run"   # placeholder; the live cell below captures a real one
for label, opts in {
    "continue latest": ClaudeAgentOptions(continue_conversation=True),
    "resume": ClaudeAgentOptions(resume=session_id),
    "fork": ClaudeAgentOptions(resume=session_id, fork_session=True, max_turns=5),
}.items():
    print(f"{label:16} continue={opts.continue_conversation!s:5} resume={opts.resume} fork_session={opts.fork_session}")


def session_strategy(earlier_context_accurate: bool, files_changed: bool, stale_fraction: float,
                     want_alternatives: bool) -> str:
    if stale_fraction > 0.5:
        return "new session seeded with a structured summary (most observations are stale)"
    if want_alternatives:
        return "fork_session=True once per alternative"
    if files_changed:
        return "resume, and tell the agent exactly what changed"
    return "resume" if earlier_context_accurate else "new session with a summary"


def fresh_session_prompt(summary: dict, changed: list[str], task: str) -> str:
    return (f"<prior_summary>\n{json.dumps(summary, indent=2)}\n</prior_summary>\n"
            f"<changed_since>{', '.join(changed) or 'nothing'}</changed_since>\n"
            f"Task: {task}\nTreat the summary as a starting point and re-check anything listed as changed.")


print(session_strategy(True, False, 0.0, True))
print(session_strategy(True, True, 0.8, False))
print(fresh_session_prompt({"constraints": ["10k rps", "p99 < 50 ms"], "decision": "cache needed"},
                           ["catalog schema"], "Propose a caching design."))

### Named sessions

In the CLI you name a session with `claude -n <name>` and return with `claude --resume <name>`. In the Python SDK, `resume=` takes a session **ID**, so you attach a title with `rename_session(session_id, title)` and look it up again with `list_sessions(directory=...)`, which returns `SDKSessionInfo` entries with `custom_title` and `session_id`. The helper below works on any list of such entries. The live cell after it names a real session and finds it again.

In [ ]:
# Find a session by name (offline: fabricated SDKSessionInfo-like entries)
def find_session(sessions, title: str) -> str | None:
    """Return the session_id of the most recently modified session with this custom title."""
    matches = [s for s in sessions if (s.custom_title or "").strip().lower() == title.strip().lower()]
    return max(matches, key=lambda s: s.last_modified).session_id if matches else None


fake_sessions = [
    SimpleNamespace(session_id="11111111-aaaa-4aaa-8aaa-111111111111", custom_title="catalog-cache-investigation", last_modified=100),
    SimpleNamespace(session_id="22222222-bbbb-4bbb-8bbb-222222222222", custom_title="auth-refactor", last_modified=200),
    SimpleNamespace(session_id="33333333-cccc-4ccc-8ccc-333333333333", custom_title="catalog-cache-investigation", last_modified=300),
]
sid_by_name = find_session(fake_sessions, "catalog-cache-investigation")
print("resume by name ->", sid_by_name)
print(ClaudeAgentOptions(resume=sid_by_name).resume == sid_by_name)

In [ ]:
# (live) Capture -> resume -> fork twice -> compare. Four tiny runs, ~$0.05 total
require_live()
plain = dict(system_prompt="You are a concise software architect. Answer in at most 5 bullets.",
             max_turns=2, max_budget_usd=0.10)
first, _ = await run_query("A read-heavy product catalog API serves 10k rps with p99 < 50 ms. "
                           "List the key constraints only, no solution yet.", base_options(**plain), verbose=False)
assert first is not None, "the first run failed; see the error above"
sid = first.session_id
follow, _ = await run_query("Now propose one design.", base_options(**plain, resume=sid), verbose=False)
fork_a, _ = await run_query("Design it with a CDN + HTTP caching.", base_options(**plain, resume=sid, fork_session=True), verbose=False)
fork_b, _ = await run_query("Design it with a Redis read-through cache.", base_options(**plain, resume=sid, fork_session=True), verbose=False)
print("original:", sid, "\nresumed :", follow.session_id, "\nfork A  :", fork_a.session_id, "\nfork B  :", fork_b.session_id)
print("\nFORK A:\n", fork_a.result, "\n\nFORK B:\n", fork_b.result)

# Name the original session, then find it again by that name.
# Session transcripts live under ~/.claude/projects/<encoded cwd>/; directory= limits this to the sandbox's sessions.
rename_session(sid, "catalog-cache-investigation", directory=SANDBOX_DIR)
print("\nfound by name:", find_session(list_sessions(directory=SANDBOX_DIR), "catalog-cache-investigation") == sid)

> **Observe:** the resumed run keeps the original ID, while each fork gets a new one. Both forks start from the same analyzed constraints, and neither sees the other's design.
>
> **Try it:** start a brand-new session with `fresh_session_prompt(...)`, using a summary you write from `first.result`, and compare its design with the forks.

## 3.7 Human approval for irreversible actions

Some actions are fine **sometimes**, but a person should confirm them first: sending an email, deploying, paying. The SDK resolves every tool request in a fixed order: **hooks → deny rules → ask rules → permission mode → allow rules → `can_use_tool`**. A call that an earlier step approves never reaches your approval callback, and `dontAsk` never calls it at all. A hook `allow` does not skip deny or ask rules. Set `permission_mode` explicitly, because an unset mode may start in `auto`. [Theory](README.md#37-human-approval-for-irreversible-actions)

In [ ]:
# The permission evaluation order (offline teaching model of the documented flow, simplified)
@dataclasses.dataclass
class Policy:
    mode: str = "default"            # default | dontAsk | acceptEdits | bypassPermissions | plan
    allow: tuple[str, ...] = ()      # allowed_tools / settings allow rules
    deny: tuple[str, ...] = ()       # disallowed_tools: bare names, or scoped rules like "Bash(rm *)"
    ask: tuple[str, ...] = ()        # settings ask rules
    callback: Any = None             # can_use_tool


EDIT_TOOLS = {"Edit", "Write", "NotebookEdit"}


def rule_matches(rule: str, tool_name: str, tool_input: dict) -> bool:
    if "(" not in rule:
        return rule == tool_name
    name, pattern = rule[:-1].split("(", 1)
    target = tool_input.get("command") or tool_input.get("file_path") or ""
    return name == tool_name and fnmatch(target, pattern)


async def decide(tool_name: str, tool_input: dict, policy: Policy, hook_decision: str | None = None) -> str:
    # 1. hooks: a deny wins everywhere
    if hook_decision == "deny":
        return "BLOCKED by a PreToolUse hook"
    # 2. deny rules: evaluated even after a hook allow, and in every mode
    for rule in policy.deny:
        if rule == tool_name:
            return "REMOVED: bare deny rule, Claude never sees this tool"
        if rule_matches(rule, tool_name, tool_input):
            return f"BLOCKED by deny rule {rule!r}"
    # 3. ask rules (or a hook "ask") go straight to the callback
    forced_ask = hook_decision == "ask" or any(rule_matches(r, tool_name, tool_input) for r in policy.ask)
    if not forced_ask:
        if hook_decision == "allow":
            return "APPROVED by the hook (deny and ask rules were still checked first)"
        # 4. permission mode
        if policy.mode == "bypassPermissions":
            return "APPROVED by bypassPermissions"
        if policy.mode == "acceptEdits" and tool_name in EDIT_TOOLS:
            return "APPROVED by acceptEdits"
        # 5. allow rules (plan mode never auto-approves edits)
        if not (policy.mode == "plan" and tool_name in EDIT_TOOLS):
            if any(rule_matches(r, tool_name, tool_input) for r in policy.allow):
                return "APPROVED by an allow rule: can_use_tool never runs"
            if tool_name in NEEDS_NO_APPROVAL:
                return "runs: needs no approval"
    # 6. can_use_tool
    if policy.mode == "dontAsk":
        return "DENIED: dontAsk never asks"
    if policy.callback is None:
        return "DENIED: nothing to ask (no can_use_tool)"
    result = await policy.callback(tool_name, tool_input, ToolPermissionContext(tool_use_id="toolu_sim"))
    if isinstance(result, PermissionResultAllow):
        edited = result.updated_input not in (None, tool_input)
        return "APPROVED by can_use_tool" + (f" with edited input {result.updated_input}" if edited else "")
    return f"DENIED by can_use_tool: {result.message}"


print("Simplified: leaves out auto mode, critical-path removals, and tools that always require user interaction.")

In [ ]:
# A can_use_tool approval gate with a scripted reviewer (offline: the callback is called directly)
REVIEW_LOG: list[str] = []
SCRIPTED_REVIEWS = {
    "mcp__mail__send": {"approved": True, "edit": {"cc": "support-lead@example.com"}},
    "mcp__deploy__release": {"approved": False, "reason": "release freeze until Monday"},
}


async def ask_reviewer(tool_name: str, tool_input: dict) -> dict:
    """Stand-in for your review UI (web form, Slack, ticket). The callback simply awaits the answer."""
    return SCRIPTED_REVIEWS.get(tool_name, {"approved": False, "reason": "no reviewer is assigned to this tool"})


async def approval_gate(tool_name: str, input_data: dict, context: ToolPermissionContext):
    REVIEW_LOG.append(f"{tool_name} ({context.tool_use_id})")
    review = await ask_reviewer(tool_name, input_data)
    if review["approved"]:
        return PermissionResultAllow(updated_input={**input_data, **review.get("edit", {})})   # a new dict
    return PermissionResultDeny(message=f"A reviewer rejected {tool_name}: {review['reason']}. "
                                        "Tell the user and propose an alternative.")


APPROVAL_POLICY = Policy(
    mode="default",
    allow=("Read", "mcp__crm__lookup_customer"),              # pre-approved, read-only
    deny=("Bash(rm *)", "mcp__crm__delete_customer"),        # never, in any mode
    callback=approval_gate,                                  # everything else asks a human
)
EMAIL = {"to": "ana@example.com", "body": "Your refund is on its way."}
cases = [
    ("Read", {"file_path": "notes.md"}, None),
    ("mcp__crm__lookup_customer", {"id": "C-7"}, None),
    ("mcp__mail__send", EMAIL, None),
    ("mcp__deploy__release", {"version": "2.4.0"}, None),
    ("Bash", {"command": "rm -rf build"}, "allow"),          # a hook allow can't override a deny rule
    ("mcp__crm__delete_customer", {"id": "C-7"}, None),
]
for name, tool_input, hook in cases:
    print(f"{name:28} hook={hook!s:5} -> {await decide(name, tool_input, APPROVAL_POLICY, hook)}")

MISTAKE = dataclasses.replace(APPROVAL_POLICY, allow=(*APPROVAL_POLICY.allow, "mcp__mail__send"))
print("\npre-approved by mistake      ->", await decide("mcp__mail__send", EMAIL, MISTAKE))
print("same, plus a hook 'ask'      ->", await decide("mcp__mail__send", EMAIL, MISTAKE, hook_decision="ask"))
print("dontAsk                      ->", await decide("mcp__mail__send", EMAIL, dataclasses.replace(APPROVAL_POLICY, mode="dontAsk")))
print("bypass + scoped deny         ->", await decide("Bash", {"command": "rm -rf /tmp/x"}, Policy(mode="bypassPermissions", deny=("Bash(rm *)",))))
print("bypass, no deny              ->", await decide("Bash", {"command": "rm -rf /tmp/x"}, Policy(mode="bypassPermissions")))
print("\nreviewer saw:", REVIEW_LOG)

> **Observe:** the reviewer saw the email twice (the normal case and the hook `ask` case) and the deploy once. It never saw the pre-approved case. When the email tool was added to `allow`, the callback never ran, which is the classic approval bug. A hook `"ask"` restores the review even with the allow rule in place. The scoped deny `Bash(rm *)` held under `bypassPermissions`, and without it `bypassPermissions` approved `rm -rf`. The approved email went out with the reviewer's edit (`cc`), and Claude isn't told about the edit.

In [ ]:
# (live) can_use_tool in a real run: refunds wait for a (simulated) reviewer. Capped at $0.25
require_live()
REFUNDS_ISSUED.clear(); REVIEW_LOG.clear()


async def reviewer_live(tool_name: str, input_data: dict, context: ToolPermissionContext):
    REVIEW_LOG.append(f"{tool_name} {json.dumps(input_data)}")
    try:
        amount = float(input_data.get("amount"))
    except (TypeError, ValueError):
        amount = float("inf")
    if tool_name == "mcp__support__process_refund" and amount <= 200:
        return PermissionResultAllow(updated_input=input_data)
    return PermissionResultDeny(message="The reviewer declined this action. Explain that to the customer.")


async def keep_stream_open(input_data, tool_use_id, context):   # the documented Python workaround for can_use_tool
    return {"continue_": True}


approval_opts = base_options(
    permission_mode="default",            # the callback only runs in a mode that asks (never in dontAsk)
    system_prompt="You are a support agent. Verify the customer with get_customer, then resolve the request.",
    mcp_servers={"support": support_server},
    allowed_tools=["mcp__support__get_customer", "mcp__support__lookup_order"],   # NOT process_refund
    can_use_tool=reviewer_live,
    hooks={"PreToolUse": [HookMatcher(matcher=None, hooks=[keep_stream_open])]},
)
# ClaudeSDKClient keeps the streaming control channel open, which can_use_tool needs.
# Expect a CanUseToolShadowedWarning naming the pre-approved tools: that is the point, they skip the reviewer.
async with ClaudeSDKClient(options=approval_opts) as sdk_client:
    await sdk_client.query("I'm ana@example.com. Order A-1001 arrived damaged. Please refund it in full.")
    async for m in sdk_client.receive_response():
        show(m)
print("\nreviewed:", REVIEW_LOG)
print("refunds issued:", REFUNDS_ISSUED)

> **Observe:** only `process_refund` reached the reviewer. The reads were pre-approved, and `ToolSearch` needs no approval. A-1001 ($129) is under the reviewer's $200 limit, so the refund runs. Ask for A-1002 ($940) instead and the reviewer declines, so Claude explains instead of refunding.
>
> **Try it:** if a real review can take hours, don't keep the process waiting in the callback. Return `permissionDecision: "defer"` from a `PreToolUse` hook instead. The run ends with `stop_reason == "tool_deferred"`, and you resume the session after the reviewer decides. This works only when Claude made a single tool call in that turn.

## Mini-project: research coordinator with a spend gate

Build a **briefing desk** that combines every concept in this chapter:

```
User ──> coordinator (main agent: Agent tool only, plus the briefing MCP server)
           ├─ Agent ──> market-researcher   tools: search_corpus, get_document
           └─ Agent ──> data-buyer          tools: list_paid_reports, purchase_report, request_purchase_approval
hooks:  PreToolUse  purchase_report        -> deny > $500 (checks the REAL price), redirect to approval
        PostToolUse search_corpus|get_document -> normalize dates to YYYY-MM-DD
        SubagentStart / SubagentStop       -> audit log
caps:   max_budget_usd, max_turns, CLAUDE_CODE_MAX_SUBAGENT_SPAWN_DEPTH=1
```

The spend gate plays the role of the README's refund limit: one violation costs real money, so the rule lives in a hook and not in the prompt.

In [ ]:
# Paid-report tools + the briefing server (offline definition)
PAID_REPORTS = {
    "R-150": {"title": "AI in music licensing 2026", "price_usd": 150},
    "R-900": {"title": "Global creative-AI market sizing 2026", "price_usd": 900},
}
PURCHASES: list[dict] = []
APPROVAL_REQUESTS: list[dict] = []
SPEND_LIMIT_USD = 500


@tool("list_paid_reports", "List paid analyst reports with id, title and price_usd.",
      {"type": "object", "properties": {}},
      annotations=ToolAnnotations(readOnlyHint=True))
async def list_paid_reports(args: dict[str, Any]) -> dict[str, Any]:
    return as_result([{"report_id": k, **v} for k, v in PAID_REPORTS.items()])


@tool("purchase_report", "Buy one paid report. Inputs: report_id, price_usd.", {"report_id": str, "price_usd": float})
async def purchase_report(args: dict[str, Any]) -> dict[str, Any]:
    PURCHASES.append(args)
    return as_result({"purchased": args["report_id"], "summary": "Market sized at $X bn (demo data)."})


@tool("request_purchase_approval", "Ask a human to approve a purchase. Inputs: report_id, price_usd, justification.",
      {"report_id": str, "price_usd": float, "justification": str})
async def request_purchase_approval(args: dict[str, Any]) -> dict[str, Any]:
    APPROVAL_REQUESTS.append(args)
    return as_result({"approval_ticket": f"PA-{len(APPROVAL_REQUESTS):03d}", "status": "pending_human_review"})


briefing_server = create_sdk_mcp_server(name="briefing", version="1.0.0", tools=[
    search_corpus, get_document, list_paid_reports, purchase_report, request_purchase_approval])

In [ ]:
# Agents, hooks and options for the briefing desk (offline definition)
B = "mcp__briefing__"

BRIEFING_AGENTS = {
    "market-researcher": AgentDefinition(
        description=("Searches the free research corpus for ONE assigned creative-industry area and returns "
                     "cited findings. Use for all fact-finding. Never buys anything."),
        prompt=("Research only the area you are given. Return a JSON list of {claim, doc_id, url, published}. "
                "Return [] and say 'no sources' if nothing matches."),
        tools=[f"{B}search_corpus", f"{B}get_document"],
        model=SUBAGENT_MODEL, maxTurns=6,
    ),
    "data-buyer": AgentDefinition(
        description=("Buys a paid analyst report when free sources lack a needed number, within the spend policy. "
                     "Use only when the coordinator names a report_id."),
        prompt=("Check the price with list_paid_reports, then try purchase_report. If a purchase is denied, call "
                "request_purchase_approval with a one-line justification. Return JSON {report_id, outcome, ticket}."),
        tools=[f"{B}list_paid_reports", f"{B}purchase_report", f"{B}request_purchase_approval"],
        model=SUBAGENT_MODEL, maxTurns=4,
    ),
}


async def enforce_spend_limit(input_data, tool_use_id, context):        # PreToolUse on purchase_report
    tool_input = input_data["tool_input"]
    report = PAID_REPORTS.get(tool_input.get("report_id"))
    if report is None:
        return deny("Unknown report_id. Call mcp__briefing__list_paid_reports first.")
    try:
        claimed = float(tool_input.get("price_usd"))
    except (TypeError, ValueError):
        return deny("price_usd is missing or not a number.")
    price = max(claimed, report["price_usd"])      # never trust the model's number alone
    if price > SPEND_LIMIT_USD:
        return deny(f"Purchases over ${SPEND_LIMIT_USD} need human approval. Call "
                    "mcp__briefing__request_purchase_approval with report_id, price_usd and a justification.")
    return {}


BRIEFING_HOOKS = {
    "PreToolUse": [HookMatcher(matcher=f"{B}purchase_report", hooks=[enforce_spend_limit])],
    "PostToolUse": [HookMatcher(matcher=f"{B}search_corpus|{B}get_document", hooks=[normalize_tool_output])],
    "SubagentStart": [HookMatcher(hooks=[log_subagent])],
    "SubagentStop": [HookMatcher(hooks=[log_subagent])],
}

BRIEFING_COORDINATOR = (
    "You run a briefing desk. Goal: a one-page, fully cited briefing.\n"
    "- Cover ALL major creative areas; give each market-researcher ONE distinct area, in parallel.\n"
    "- Put everything a subagent needs in its prompt; subagents see nothing else.\n"
    "- Use data-buyer only for a specific report_id when a needed number is missing.\n"
    "- Report gaps, pending approvals and conflicting sources explicitly."
)

briefing_opts = base_options(
    system_prompt=BRIEFING_COORDINATOR,
    agents=BRIEFING_AGENTS,
    mcp_servers={"briefing": briefing_server},
    allowed_tools=["Agent", f"{B}*"],
    hooks=BRIEFING_HOOKS,
    max_turns=20,
    max_budget_usd=0.60,
    env={"CLAUDE_CODE_MAX_SUBAGENT_SPAWN_DEPTH": "1", "CLAUDE_CODE_MAX_CONCURRENT_SUBAGENTS": "4"},
)
print(audit_agents(BRIEFING_AGENTS) or "agents OK", "| budget:", briefing_opts.max_budget_usd)

In [ ]:
# Self-test the deterministic parts before spending anything (offline)
checks = {
    "R-900 at its real price is denied":
        (await simulate_pre_tool_use(BRIEFING_HOOKS, f"{B}purchase_report", {"report_id": "R-900", "price_usd": 900}))[0] == "deny",
    "lying about the price doesn't help":
        (await simulate_pre_tool_use(BRIEFING_HOOKS, f"{B}purchase_report", {"report_id": "R-900", "price_usd": 90}))[0] == "deny",
    "R-150 passes the gate":
        (await simulate_pre_tool_use(BRIEFING_HOOKS, f"{B}purchase_report", {"report_id": "R-150", "price_usd": 150}))[0].startswith("no decision"),
    "unknown report is denied":
        (await simulate_pre_tool_use(BRIEFING_HOOKS, f"{B}purchase_report", {"report_id": "R-1", "price_usd": 1}))[0] == "deny",
    "researcher can't buy":
        not any("purchase" in t for t in BRIEFING_AGENTS["market-researcher"].tools),
    "buyer can't search":
        f"{B}search_corpus" not in BRIEFING_AGENTS["data-buyer"].tools,
}
raw = await search_corpus.handler({"query": "music voice royalty"})
[post] = await simulate_post_tool_use(BRIEFING_HOOKS, f"{B}search_corpus", raw)   # the MCP result object
dates = [d["published"] for d in json.loads(tool_text(post["hookSpecificOutput"]["updatedToolOutput"]))["results"]]
checks["dates normalized to YYYY-MM-DD"] = all(re.fullmatch(r"\d{4}-\d{2}-\d{2}", d) for d in dates)
for name, ok in checks.items():
    print("✓" if ok else "✗", name)
assert all(checks.values()), "fix the failing check before running live"

In [ ]:
# (live) Run the briefing desk. Hard-capped at $0.60
require_live()
PURCHASES.clear(); APPROVAL_REQUESTS.clear(); AUDIT_LOG.clear()
result, msgs = await run_query(
    "Prepare a one-page briefing on how AI is changing the creative industries. Cover every major area. "
    "The corpus has no market-size numbers, so get analyst report R-900 for those.",
    briefing_opts, verbose=False)
print(trace(msgs))
print("purchases:", PURCHASES)
print("approval requests:", APPROVAL_REQUESTS)
print("audit:", *AUDIT_LOG, sep="\n  ")
if result:
    print("\n", handle_result(result))

> **Observe:** `PURCHASES` stays empty for R-900, the data-buyer files an approval request instead, and the briefing reports the pending approval as a gap. The audit log shows each subagent's start and stop. The coordinator never saw the researchers' raw search calls.
>
> **Try it:**
> 1. Ask for R-150 instead and confirm the purchase goes through.
> 2. Add a `PreToolUse` hook with matcher `f"{B}.*"` that denies any `mcp__briefing__*` call whose `input_data` has no `agent_id`, so the coordinator **must** delegate (hub-and-spoke enforced in code).
> 3. Set `max_budget_usd=0.05` and handle `error_max_budget_usd` gracefully.

## Recap

- The loop ends on `stop_reason == "end_turn"` (Client SDK) or on `ResultMessage.subtype` (Agent SDK). Caps are guardrails, and text matching is a bug.
- `@tool` + `create_sdk_mcp_server` give Claude your functions as `mcp__<server>__<tool>`, and you can unit-test handlers without the model.
- `AgentDefinition(description, prompt, tools, model, maxTurns)` goes in `agents={name: ...}`. Omitting `tools` grants everything.
- `allowed_tools` only auto-approves. Restrict with `disallowed_tools`, `AgentDefinition.tools`, `dontAsk`, or hooks.
- The coordinator owns decomposition, context passing, aggregation, and error handling. Spawn subagents with the `Agent` tool (also match `Task`), in parallel when the subtasks are independent.
- `PreToolUse` hooks enforce money, identity, and compliance rules deterministically, and `PostToolUse` hooks normalize data centrally.
- `resume` continues a session and `fork_session=True` branches it. Name sessions (`claude -n`, or `rename_session` in the SDK) to find them again. When most earlier context is stale, start fresh from a summary.
- Approvals follow a fixed order: hooks, deny rules, ask rules, mode, allow rules, `can_use_tool`. Never pre-approve a tool you want reviewed, set `permission_mode` explicitly, and keep must-never rules in deny rules or hooks.

Next: [homework](02_homework.ipynb)